In [ ]:
#Haar system

In [ ]:
import math
import random
import matplotlib.pyplot as plt
import time
import numpy as np

class Mirror_descent_Banach():
    def __init__(self,noise,para,s,p,q,theta,tau,gamma0,N):
        self.s = s
        self.p = p
        self.pp = p/(p-1)
        self.q = q
        self.Ln = 1
        self.fn = np.zeros(1)
        self.f_average_n = np.zeros(1)
        self.theta = theta
        self.gamma0 = gamma0 #Initial step size
        self.tau = tau #Step decay rate
        self.Num = np.zeros(1)
        self.epoch = 1 #  Current number of iterations
        self.Hn = 0
        self.a0 = 2*np.cos(1.5*math.pi*np.arccos(1-abs(0-0.5))) + (2*0)**2 - 3*abs(0-1/4)**(2/3) + 3*np.cos(5*0) - np.exp(3*0) 
        self.a1 = 2*np.cos(1.5*math.pi*np.arccos(1-abs(1-0.5))) + (2*1)**2 - 3*abs(1-1/4)**(2/3) + 3*np.cos(5*1) - np.exp(3*1)
        self.N = N
        self.noise = noise
        self.para = para

    def cal_single_basis(self,n,k,x):
        if n == -1:
            y = 1
        else:
            a0 = 2**n
            a1 = 1/a0
            a0_sqrt = np.sqrt(a0)
            if x< (k-1)*a1:
                y = 0
            elif x < (2*k-1)*a1/2:
                y = a0_sqrt
            elif x < k*a1:
                y = -a0_sqrt
            else:
                y = 0
            if k == 1 and x == 1:
                y = -a0_sqrt
        return y
    
    def cal_basis_function(self, X):
        Output = np.zeros(self.Ln)
        Output[0] = 1.0
        if self.Ln == 1:
            return Output
        x = float(X)
        if x == 1.0:
            x = np.nextafter(1.0, 0.0)
        scale = 1
        while scale < self.Ln:
            position = scale * x
            integer_part = int(position)
            k = integer_part + 1
            i = scale + k
            if i <= self.Ln:
                local_position = position - integer_part

                if local_position < 0.5:
                    Output[i - 1] = math.sqrt(scale)
                else:
                    Output[i - 1] = -math.sqrt(scale)
            scale *= 2
        return Output
    
    def fn_cal(self,X):
        Output = self.cal_basis_function(X)
        return np.sum(self.fn*Output)
        
    def f_average_n_cal(self,X):
        Output = self.cal_basis_function(X)
        return np.sum(self.f_average_n*Output)

    def data_noise(self):
        c = self.para
        W = np.random.exponential()
        U = np.random.uniform(-math.pi/2,math.pi/2)
        epsilon = 2*(np.sin(c*U)/(np.cos(U)**(1/c)))*(np.cos(U-c*U)/W)**((1-c)/c)
        return epsilon 
        
    def data(self):
        X = random.random()
        Y = self.f_rho_cal(X)
        if self.noise == 'empty':
            Z = Y
        elif self.noise == 'uniform':
            Z = Y + random.uniform(-0.05,0.05)
        elif self.noise == 'gauss':
            Z = Y + random.gauss(0,0.5)
        elif self.noise == 'heavy-tail':
            Z = Y + self.data_noise() 
        return (X,Z,Y)
    
    def f_rho_cal(self,x):
        return  np.exp(0.7*np.cos(2*np.pi*x))*np.sin(2*np.pi*x) + 0.5*np.sin(6*np.pi*x) - 0.3*np.sin(10*np.pi*x) + 0.1*np.sin(14*np.pi*x) + 2*abs(np.cos(2*np.pi*x))**(7/2) - 0.5*abs(np.sin(2*np.pi*(x-0.25)))**(7/2)

    def fuhao(self):
        if self.p-2 < 0:
            b = 1
        else:
            b = 0
        return b
    
    def iteration(self,X,Y):
        if self.epoch**self.theta > self.Ln:
            self.Ln += 1
            self.Num = np.append(self.Num,self.Ln-1)
            self.fn = np.append(self.fn,0)
            self.f_average_n = np.append(self.f_average_n,0)
        self.gamma = self.gamma0*(self.epoch**(-self.tau))
        Output = self.cal_basis_function(X)
        fX = np.sum(self.fn*Output)
        w_f = self.q*(abs(fX-Y)**(self.q-1))*np.sign(fX-Y)
        J_list = np.array([ j**(2*self.s*self.p) for j in range(1,self.Ln+1)])
        B = (np.sum(J_list*(np.abs(self.fn)**self.p))+10**(-8))**(max(0,2-self.p)/self.p)
        Alpha = B*(np.abs(self.fn)**(self.p-1))*np.sign(self.fn)*J_list-self.gamma*w_f*Output
        J_list_new = np.array([ j**(-2*self.s*self.pp) for j in range(1,self.Ln+1)])
        A = (np.sum(J_list_new*(np.abs(Alpha)**self.pp))+10**(-8))**(min(0,self.p-2)/self.p)
        self.fn = A*J_list_new*(np.abs(Alpha)**(1/(self.p-1)))*np.sign(Alpha)
        if self.epoch >= self.sum_epoch/2:
            self.f_average_n = self.n/(self.n+1)*self.f_average_n + 1/(self.n+1)*self.fn
            self.n += 1
        #print(A,B)

    
    def gauss_legendre_01(self, n_x):
        """
        Gauss--Legendre nodes and weights on [0,1].
        """
        t, w = np.polynomial.legendre.leggauss(n_x)
        x = 0.5 * (t + 1.0)
        w = 0.5 * w
        return x, w


    def sample_abs_noise(self, n_noise):
        """
        Sample eta_k = |xi_k| from the noise distribution.

        Because self.data() returns (X, noisy_value, clean_value)
        after local assignment X, Y, Z = self.data(),
        the noise is Y - Z.
        """
        eta_list = []
        for _ in range(n_noise):
            if self.noise == 'empty':
                xi = 0
            elif self.noise == 'uniform':
                xi = random.uniform(-0.05,0.05)
            elif self.noise == 'gauss':
                xi = random.gauss(0,0.5)
            elif self.noise == 'heavy-tail':
                xi = self.data_noise()
            eta_list.append(abs(xi))
        return np.array(eta_list, dtype=float)


    def test_lq_gauss_symmetric(self, n_x=1024, n_noise=4096, chunk=1024):
        """
        Evaluate errors using:
        1. Gauss--Legendre quadrature for x in [0,1];
        2. symmetric noise pairing eta = |xi|.

        It computes both:

        clean Lq error:
            int_0^1 |f(x)-f_rho(x)|^q dx,

        and symmetric noisy excess risk:
            int_0^1 E_eta [
                (|d(x)-eta|^q + |d(x)+eta|^q)/2 - eta^q
            ] dx,

        where d(x)=f(x)-f_rho(x).
        """

        q = self.q

        # Gauss--Legendre points for x
        xs, ws = self.gauss_legendre_01(n_x)

        # Symmetric noise samples eta=|xi|
        eta = self.sample_abs_noise(n_noise)

        # Evaluate f, averaged f, and f_rho on Gauss nodes
        f_vals = np.array([self.fn_cal(x) for x in xs], dtype=float)
        fave_vals = np.array([self.f_average_n_cal(x) for x in xs], dtype=float)
        frho_vals = np.array([self.f_rho_cal(x) for x in xs], dtype=float)

        d_fn = f_vals - frho_vals
        d_ave = fave_vals - frho_vals

        # Clean Lq errors:
        # int |f-f_rho|^q dx
        clean_fn = np.sum(ws * np.abs(d_fn)**q)
        clean_ave = np.sum(ws * np.abs(d_ave)**q)

        # Symmetric noisy excess risks:
        # int E_eta A_q(d(x), eta) dx
        # A_q(d, eta) = (|d-eta|^q + |d+eta|^q)/2 - eta^q
        A_fn_sum = np.zeros(n_x, dtype=float)
        A_ave_sum = np.zeros(n_x, dtype=float)

        # Also compute noisy risks themselves if needed:
        # int E_eta (|d-eta|^q + |d+eta|^q)/2 dx
        risk_fn_sum = np.zeros(n_x, dtype=float)
        risk_ave_sum = np.zeros(n_x, dtype=float)

        d_fn_mat = d_fn.reshape(-1, 1)
        d_ave_mat = d_ave.reshape(-1, 1)

        for start in range(0, n_noise, chunk):
            end = min(start + chunk, n_noise)
            eta_chunk = eta[start:end].reshape(1, -1)

            eta_q = eta_chunk**q

            risk_fn_chunk = 0.5 * (
                np.abs(d_fn_mat - eta_chunk)**q
                +
                np.abs(d_fn_mat + eta_chunk)**q
            )

            risk_ave_chunk = 0.5 * (
                np.abs(d_ave_mat - eta_chunk)**q
                +
                np.abs(d_ave_mat + eta_chunk)**q
            )

            A_fn_chunk = risk_fn_chunk - eta_q
            A_ave_chunk = risk_ave_chunk - eta_q

            # Mathematically A_q >= 0.
            # This only removes tiny negative values caused by floating-point cancellation.
            A_fn_chunk = np.maximum(A_fn_chunk, 0.0)
            A_ave_chunk = np.maximum(A_ave_chunk, 0.0)

            risk_fn_sum += risk_fn_chunk.sum(axis=1)
            risk_ave_sum += risk_ave_chunk.sum(axis=1)

            A_fn_sum += A_fn_chunk.sum(axis=1)
            A_ave_sum += A_ave_chunk.sum(axis=1)

        risk_fn = np.sum(ws * (risk_fn_sum / n_noise))
        risk_ave = np.sum(ws * (risk_ave_sum / n_noise))

        excess_fn = np.sum(ws * (A_fn_sum / n_noise))
        excess_ave = np.sum(ws * (A_ave_sum / n_noise))

        '''
        print(
            'epoch:', self.epoch,
            'Ln:', self.Ln,
            'clean_fn:', clean_fn,
            'clean_ave:', clean_ave,
            'risk_fn:', risk_fn,
            'risk_ave:', risk_ave,
            'excess_fn:', excess_fn,
            'excess_ave:', excess_ave
        )'''

        return {
            'clean_fn': clean_fn,
            'clean_ave': clean_ave,
            'risk_fn': risk_fn,
            'risk_ave': risk_ave,
            'excess_fn': excess_fn,
            'excess_ave': excess_ave,
        }

    
    def algorithm(self,epoch):
        S_ave = 0
        S_estimator = 0
        self.sum_epoch = epoch
        self.n = 0
        for i in range(epoch):
            self.epoch = i+1
            X,Y,Z = self.data()
            self.iteration(X,Y)
        result = self.test_lq_gauss_symmetric(
            n_x=512,
                n_noise=1024,
            chunk=512
        )
        return (result['excess_ave'],result['excess_fn'],result['clean_ave'],result['clean_fn'])


In [ ]:
s = 0.5
p = 1.35
q = 1.25
theta = 0.35
tau = 0.5
gamma0 = 5.0
noise = 'heavy-tail'
para = 1.36
N = [ int(10**(i/5)) for i in range(1,26)]
SS_ave = []
SS_estimator = []
repeat = 25
for i in range(repeat):
    S1 = []
    S2 = []
    for n in N:
        mirror_descent_Banach = Mirror_descent_Banach(noise,para,s,p,q,theta,tau,gamma0+i*10**(-8),N)
        s1,s2,s3,s4 = mirror_descent_Banach.algorithm(n)
        #print(n,' ',s1,s2,s3,s4)
        S1.append(s1)
        S2.append(s2)
    print(i)
    SS_ave.append(S1[:])
    SS_estimator.append(S2[:])

s_ave = []
s_estimator = []
for i in range(len(S1)):
    s_ave.append(sum([ SS_ave[j][i] for j in range(repeat)])/repeat)
    s_estimator.append(sum([ SS_estimator[j][i] for j in range(repeat)])/repeat)
print('ave=',s_ave)
print('fn=',s_estimator)
    
    

In [ ]:
s = 0.5
p = 1.75
q = 1.5
theta = 0.35
tau = 0.5
gamma0 = 2.5
noise = 'heavy-tail'
para = 1.76
N = [ int(10**(i/5)) for i in range(1,26)]
SS_ave = []
SS_estimator = []
repeat = 25
for i in range(repeat):
    S1 = []
    S2 = []
    for n in N:
        mirror_descent_Banach = Mirror_descent_Banach(noise,para,s,p,q,theta,tau,gamma0+i*10**(-8),N)
        s1,s2,s3,s4 = mirror_descent_Banach.algorithm(n)
        #print(n,' ',s1,s2,s3,s4)
        S1.append(s1)
        S2.append(s2)
    print(i)
    SS_ave.append(S1[:])
    SS_estimator.append(S2[:])

s_ave = []
s_estimator = []
for i in range(len(S1)):
    s_ave.append(sum([ SS_ave[j][i] for j in range(repeat)])/repeat)
    s_estimator.append(sum([ SS_estimator[j][i] for j in range(repeat)])/repeat)
print('ave=',s_ave)
print('fn=',s_estimator)
    
    

In [ ]:
s = 0.4
p = 2.5
q = 2.0
theta = 0.35
tau = 0.5
gamma0 = 2.0
noise = 'uniform'
para = 1.76
N = [ int(10**(i/5)) for i in range(1,26)]
SS_ave = []
SS_estimator = []
repeat = 25
for i in range(repeat):
    S1 = []
    S2 = []
    for n in N:
        mirror_descent_Banach = Mirror_descent_Banach(noise,para,s,p,q,theta,tau,gamma0+i*10**(-8),N)
        s1,s2,s3,s4 = mirror_descent_Banach.algorithm(n)
        #print(n,' ',s1,s2,s3,s4)
        S1.append(s1)
        S2.append(s2)
    print(i)
    SS_ave.append(S1[:])
    SS_estimator.append(S2[:])

s_ave = []
s_estimator = []
for i in range(len(S1)):
    s_ave.append(sum([ SS_ave[j][i] for j in range(repeat)])/repeat)
    s_estimator.append(sum([ SS_estimator[j][i] for j in range(repeat)])/repeat)
print('ave=',s_ave)
print('fn=',s_estimator)


In [ ]:
s = 0.4
p = 4.0
q = 3.0
theta = 0.35
tau = 0.5
gamma0 = 3.0
noise = 'uniform'
para = 1.76
N = [ int(10**(i/5)) for i in range(1,26)]
SS_ave = []
SS_estimator = []
repeat = 25
for i in range(repeat):
    S1 = []
    S2 = []
    for n in N:
        mirror_descent_Banach = Mirror_descent_Banach(noise,para,s,p,q,theta,tau,gamma0+i*10**(-8),N)
        s1,s2,s3,s4 = mirror_descent_Banach.algorithm(n)
        #print(n,' ',s1,s2,s3,s4)
        S1.append(s1)
        S2.append(s2)
    print(i)
    SS_ave.append(S1[:])
    SS_estimator.append(S2[:])

s_ave = []
s_estimator = []
for i in range(len(S1)):
    s_ave.append(sum([ SS_ave[j][i] for j in range(repeat)])/repeat)
    s_estimator.append(sum([ SS_estimator[j][i] for j in range(repeat)])/repeat)
print('ave=',s_ave)
print('fn=',s_estimator)
    
    

In [ ]:
'p=1'

In [ ]:
import math
import random
import matplotlib.pyplot as plt
import time
import numpy as np

class Mirror_descent_Banach():
    def __init__(self,noise,para,s,theta,gamma0,N,L=10.):
        self.s = s
        self.p = 1
        self.q = 1
        self.N = N
        self.theta = theta
        self.Ln = math.ceil(self.N**self.theta)
        self.L = L
        initial_alpha = (3*self.L/(math.pi**2)* np.array([ j**(-2*self.s-2) for j in range(1,self.Ln+1)]))
        self.alpha_plus = initial_alpha.copy()
        self.alpha_minus = initial_alpha.copy()
        self.fn = self.alpha_plus-self.alpha_minus
        self.f_average_n = np.zeros(self.Ln)
        self.gamma0 = gamma0
        self.gamma = self.gamma0*(self.N**(-1/2))
        self.Num = np.zeros(1)
        self.epoch = 1 
        self.noise = noise
        self.para = para
        self.J_list = np.array([ j**(2*self.s) for j in range(1,self.Ln+1)])
        self.J_list_new = np.array([ j**(-2*self.s) for j in range(1,self.Ln+1)])

    def cal_single_basis(self,n,k,x):
        if n == -1:
            y = 1
        else:
            a0 = 2**n
            a1 = 1/a0
            a0_sqrt = np.sqrt(a0)
            if x< (k-1)*a1:
                y = 0
            elif x < (2*k-1)*a1/2:
                y = a0_sqrt
            elif x < k*a1:
                y = -a0_sqrt
            else:
                y = 0
            if k == 1 and x == 1:
                y = -a0_sqrt
        return y
    
    def cal_basis_function(self,X):
        Output = np.zeros(self.Ln)
        Output[0] = 1.0
        if self.Ln == 1:
            return Output
        x = float(X)
        if x == 1.0:
            x = np.nextafter(1.0,0.0)
        scale = 1
        while scale < self.Ln:
            position = scale*x
            integer_part = int(position)
            k = integer_part+1
            i = scale+k
            if i <= self.Ln:
                local_position = position-integer_part
                if local_position < 0.5:
                    Output[i-1] = math.sqrt(scale)
                else:
                    Output[i-1] = -math.sqrt(scale)
            scale *= 2
        return Output
    
    def fn_cal(self,X):
        Output = self.cal_basis_function(X)
        return np.sum(self.fn*Output)
        
    def f_average_n_cal(self,X):
        Output = self.cal_basis_function(X)
        return np.sum(self.f_average_n*Output)

    def data_noise(self):
        c = self.para
        W = np.random.exponential()
        U = np.random.uniform(-math.pi/2,math.pi/2)
        epsilon = 2*(np.sin(c*U)/(np.cos(U)**(1/c)))*(np.cos(U-c*U)/W)**((1-c)/c)
        return epsilon 
        
    def data(self):
        X = random.random()
        Y = self.f_rho_cal(X)
        if self.noise == 'empty':
            Z = Y
        elif self.noise == 'uniform':
            Z = Y + random.uniform(-0.05,0.05)
        elif self.noise == 'gauss':
            Z = Y + random.gauss(0,0.5)
        elif self.noise == 'heavy-tail':
            Z = Y + self.data_noise() 
        return (X,Z,Y)
    
    def f_rho_cal(self,x):
        return  np.exp(0.7*np.cos(2*np.pi*x))*np.sin(2*np.pi*x) + 0.5*np.sin(6*np.pi*x) - 0.3*np.sin(10*np.pi*x) + 0.1*np.sin(14*np.pi*x) + 2*abs(np.cos(2*np.pi*x))**(7/2) - 0.5*abs(np.sin(2*np.pi*(x-0.25)))**(7/2)
        
    
    def iteration(self,X,Y):
        Output = self.cal_basis_function(X)
        fX = np.sum(self.fn*Output)
        w_f = np.sign(fX-Y)
        Alpha_plus = self.alpha_plus*np.exp(-self.gamma*w_f*self.J_list_new*Output)
        Alpha_minus = self.alpha_minus*np.exp(self.gamma*w_f*self.J_list_new*Output)
        A = max(1, np.sum(self.J_list*(Alpha_plus+Alpha_minus))/self.L)
        self.alpha_plus = Alpha_plus/A
        self.alpha_minus = Alpha_minus/A
        self.fn = self.alpha_plus-self.alpha_minus
        if self.epoch > self.sum_epoch/2:
            self.f_average_n = (self.n/(self.n+1)*self.f_average_n + 1/(self.n+1)*self.fn)
            self.n += 1

    
    def gauss_legendre_01(self, n_x):
        """
        Gauss--Legendre nodes and weights on [0,1].
        """
        t, w = np.polynomial.legendre.leggauss(n_x)
        x = 0.5 * (t + 1.0)
        w = 0.5 * w
        return x, w


    def sample_abs_noise(self, n_noise):
        """
        Sample eta_k = |xi_k| from the noise distribution.

        Because self.data() returns (X, noisy_value, clean_value)
        after local assignment X, Y, Z = self.data(),
        the noise is Y - Z.
        """
        eta_list = []
        for _ in range(n_noise):
            if self.noise == 'empty':
                xi = 0
            elif self.noise == 'uniform':
                xi = random.uniform(-0.05,0.05)
            elif self.noise == 'gauss':
                xi = random.gauss(0,0.5)
            elif self.noise == 'heavy-tail':
                xi = self.data_noise()
            eta_list.append(abs(xi))
        return np.array(eta_list, dtype=float)


    def test_lq_gauss_symmetric(self, n_x=1024, n_noise=4096, chunk=1024):
        """
        Evaluate errors using:
        1. Gauss--Legendre quadrature for x in [0,1];
        2. symmetric noise pairing eta = |xi|.

        It computes both:

        clean Lq error:
            int_0^1 |f(x)-f_rho(x)|^q dx,

        and symmetric noisy excess risk:
            int_0^1 E_eta [
                (|d(x)-eta|^q + |d(x)+eta|^q)/2 - eta^q
            ] dx,

        where d(x)=f(x)-f_rho(x).
        """

        q = self.q

        # Gauss--Legendre points for x
        xs, ws = self.gauss_legendre_01(n_x)

        # Symmetric noise samples eta=|xi|
        eta = self.sample_abs_noise(n_noise)

        # Evaluate f, averaged f, and f_rho on Gauss nodes
        f_vals = np.array([self.fn_cal(x) for x in xs], dtype=float)
        fave_vals = np.array([self.f_average_n_cal(x) for x in xs], dtype=float)
        frho_vals = np.array([self.f_rho_cal(x) for x in xs], dtype=float)

        d_fn = f_vals - frho_vals
        d_ave = fave_vals - frho_vals

        # Clean Lq errors:
        # int |f-f_rho|^q dx
        clean_fn = np.sum(ws * np.abs(d_fn)**q)
        clean_ave = np.sum(ws * np.abs(d_ave)**q)

        # Symmetric noisy excess risks:
        # int E_eta A_q(d(x), eta) dx
        # A_q(d, eta) = (|d-eta|^q + |d+eta|^q)/2 - eta^q
        A_fn_sum = np.zeros(n_x, dtype=float)
        A_ave_sum = np.zeros(n_x, dtype=float)

        # Also compute noisy risks themselves if needed:
        # int E_eta (|d-eta|^q + |d+eta|^q)/2 dx
        risk_fn_sum = np.zeros(n_x, dtype=float)
        risk_ave_sum = np.zeros(n_x, dtype=float)

        d_fn_mat = d_fn.reshape(-1, 1)
        d_ave_mat = d_ave.reshape(-1, 1)

        for start in range(0, n_noise, chunk):
            end = min(start + chunk, n_noise)
            eta_chunk = eta[start:end].reshape(1, -1)

            eta_q = eta_chunk**q

            risk_fn_chunk = 0.5 * (
                np.abs(d_fn_mat - eta_chunk)**q
                +
                np.abs(d_fn_mat + eta_chunk)**q
            )

            risk_ave_chunk = 0.5 * (
                np.abs(d_ave_mat - eta_chunk)**q
                +
                np.abs(d_ave_mat + eta_chunk)**q
            )

            A_fn_chunk = risk_fn_chunk - eta_q
            A_ave_chunk = risk_ave_chunk - eta_q

            # Mathematically A_q >= 0.
            # This only removes tiny negative values caused by floating-point cancellation.
            A_fn_chunk = np.maximum(A_fn_chunk, 0.0)
            A_ave_chunk = np.maximum(A_ave_chunk, 0.0)

            risk_fn_sum += risk_fn_chunk.sum(axis=1)
            risk_ave_sum += risk_ave_chunk.sum(axis=1)

            A_fn_sum += A_fn_chunk.sum(axis=1)
            A_ave_sum += A_ave_chunk.sum(axis=1)

        risk_fn = np.sum(ws * (risk_fn_sum / n_noise))
        risk_ave = np.sum(ws * (risk_ave_sum / n_noise))

        excess_fn = np.sum(ws * (A_fn_sum / n_noise))
        excess_ave = np.sum(ws * (A_ave_sum / n_noise))

        '''
        print(
            'epoch:', self.epoch,
            'Ln:', self.Ln,
            'clean_fn:', clean_fn,
            'clean_ave:', clean_ave,
            'risk_fn:', risk_fn,
            'risk_ave:', risk_ave,
            'excess_fn:', excess_fn,
            'excess_ave:', excess_ave
        )'''

        return {
            'clean_fn': clean_fn,
            'clean_ave': clean_ave,
            'risk_fn': risk_fn,
            'risk_ave': risk_ave,
            'excess_fn': excess_fn,
            'excess_ave': excess_ave,
        }

    
    def algorithm(self):
        S_ave = 0
        S_estimator = 0
        self.sum_epoch = self.N + 1
        self.n = 0
        for i in range(1,self.sum_epoch):
            self.epoch = i
            X,Y,Z = self.data()
            self.iteration(X,Y)
        result = self.test_lq_gauss_symmetric(
            n_x=512,
                n_noise=1024,
            chunk=512
        )
        return (result['excess_ave'],result['excess_fn'],result['clean_ave'],result['clean_fn'])





In [ ]:
s = 0.25
theta = 0.35
gamma0 = 5.0
noise = 'heavy-tail'
para = 1.10
NN = [ int(10**(i/5)) for i in range(1,26)]
L = 10.
SS_ave = []
SS_estimator = []
repeat = 25
for i in range(repeat):
    S1 = []
    S2 = []
    for N in NN:
        mirror_descent_Banach = Mirror_descent_Banach(noise,para,s,theta,gamma0,N,L=10.)
        s1,s2,_,_ = mirror_descent_Banach.algorithm()
        #print(N,' ',s1,s2)
        S1.append(s1)
        S2.append(s2)
    print(i)
    SS_ave.append(S1[:])
    SS_estimator.append(S2[:])

s_ave = []
s_estimator = []
for i in range(len(S1)):
    s_ave.append(sum([ SS_ave[j][i] for j in range(repeat)])/repeat)
    s_estimator.append(sum([ SS_estimator[j][i] for j in range(repeat)])/repeat)
print('ave=',s_ave)
print('fn=',s_estimator)
